In [ ]:
HAP_BASE_URL = "http://146.56.216.100:8880/api"
HAP_APP_KEY = "19e462d66f2952af"
HAP_SIGN = "NWViY2QxMmFiOWJlZDc3YjI3ZjgwMjdmODRjMzJjYjMxYTE1MmVlZDZmZTAzNjE3YTJjODFkNmFiZDJjMDQxNg=="

JKY_APP_KEY = "91610338"
JKY_SECRET = "312a0897c9b34a8d99ed9fb61e46ae98"

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
current_dir = Path.cwd()
root_dir = current_dir.parent.parent

import sys
sys.path.append(str(root_dir))


In [ ]:
import asyncio
from apps.data_opt.components.ecerp_jky import HapConfig, HapConnection, JkyConnection, JkyConfig

import importlib
importlib.reload(sys.modules['apps.data_opt.components.ecerp_jky'])

from apps.data_opt.components.ecerp_jky import HapConfig, HapConnection, JkyConnection, JkyConfig

class TestHapConfig(HapConfig):
    BASE_URL = HAP_BASE_URL
    APP_KEY = HAP_APP_KEY
    SIGN = HAP_SIGN
    DESCRIPTION = "测试环境"
    QPS_LIMIT = 1000

sync_hap_conn = HapConnection(config=TestHapConfig)

class TestJkyConfig(JkyConfig):
    APP_KEY = JKY_APP_KEY
    APP_SECRET = JKY_SECRET

jky_conn = JkyConnection(config=TestJkyConfig, hap_conn=sync_hap_conn)


In [ ]:
from datetime import datetime

async def init_jky_data(start_time: str, end_time: str = None):
    end_time = end_time or datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    for data_source in ['$Company', '$Department', '$Staff', '$BankAccounts', '$Channel', '$GoodsCate', '$Warehouse', '$Logistic']:
        await jky_conn.data_to_hap(source_code=data_source)

    for data_source in ['~Customer', '~Sku']:
        await jky_conn.data_to_hap(source_code=data_source, slice_timerange=(start_time, end_time))

await init_jky_data()

In [ ]:
await jky_conn.data_to_hap(source_code="^Trade", other_biz={"tradeIds": ["2432337160140227072"]})
await jky_conn.data_to_hap(source_code="~Trade", slice_timerange=("2026-03-11 00:00:00", "2026-03-12 00:00:00"))
await jky_conn.data_to_hap(source_code="^Trade", slice_timerange=("2026-03-11 00:00:00", "2026-03-12 00:00:00"))
